In [58]:
import pandas as pd
import plotly.express as px
import numpy as np
import re

from enum import unique
import xgboost as xgb

import pandas as pd
import plotly.express as px
import numpy as np
import re
import nltk


In [59]:
df = pd.read_csv('train.csv')

In [60]:
df

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312
...,...,...,...,...,...,...,...,...,...,...,...
299995,299995,Adidas,Leather,Small,9.0,No,No,Tote,Blue,12.730812,129.99749
299996,299996,Jansport,Leather,Large,6.0,No,Yes,Tote,Blue,26.633182,19.85819
299997,299997,Puma,Canvas,Large,9.0,Yes,Yes,Backpack,Pink,11.898250,111.41364
299998,299998,Adidas,Nylon,Small,1.0,No,Yes,Tote,Pink,6.175738,115.89080


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [62]:
df.isna().sum()
df['Price'] = df["Price"].apply(lambda x: round(x))
df

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,69
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,81
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86
...,...,...,...,...,...,...,...,...,...,...,...
299995,299995,Adidas,Leather,Small,9.0,No,No,Tote,Blue,12.730812,130
299996,299996,Jansport,Leather,Large,6.0,No,Yes,Tote,Blue,26.633182,20
299997,299997,Puma,Canvas,Large,9.0,Yes,Yes,Backpack,Pink,11.898250,111
299998,299998,Adidas,Nylon,Small,1.0,No,Yes,Tote,Pink,6.175738,116


In [63]:
df['price_bin'] = pd.qcut(
    df['Price'],
    q=5,               # 5 ценовых сегментов
    duplicates='drop'
)


In [64]:


# =====================================================
# 1. Weight Capacity (kg) — числовой
# =====================================================
df['Weight Capacity (kg)'] = (
    df['Weight Capacity (kg)']
        .fillna(
            df.groupby(['Compartments', 'price_bin'])['Weight Capacity (kg)']
              .transform('median')
        )
        .fillna(
            df.groupby('Compartments')['Weight Capacity (kg)']
              .transform('median')
        )
        .fillna(df['Weight Capacity (kg)'].median())
        .round(0)
        .astype(int)
)


# =====================================================
# 2. Laptop Compartment — бинарный
# =====================================================
df['Laptop Compartment'] = (
    df['Laptop Compartment']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin', 'Weight Capacity (kg)']
            )['Laptop Compartment']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)']
            )['Laptop Compartment']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Laptop Compartment'].mode().iloc[0])
)


# =====================================================
# 3. Style
# =====================================================
df['Style'] = (
    df['Style']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment']
            )['Style']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)']
            )['Style']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Style'].mode().iloc[0])
)


# =====================================================
# 4. Size
# =====================================================
df['Size'] = (
    df['Size']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment', 'Style']
            )['Size']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)', 'Style']
            )['Size']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Size'].mode().iloc[0]) # Удалено .round(0)
        # Если вы все же хотите получить числовой тип, оставьте:
        .astype(int)
)


df['Material'] = (
    df['Material']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment',
                 'Style', 'Size']
            )['Material']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)', 'Style']
            )['Material']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Material'].mode().iloc[0])
)



df['Color'] = (
    df['Color']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment',
                 'Style', 'Size', 'Material']
            )['Color']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)', 'Style']
            )['Color']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Color'].mode().iloc[0])
)



df['Waterproof'] = (
    df['Waterproof']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment',
                 'Style', 'Size', 'Material', 'Color']
            )['Waterproof']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby(
                ['Compartments', 'Weight Capacity (kg)', 'Style']
            )['Waterproof']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Waterproof'].mode().iloc[0])
)


df['Brand'] = (
    df['Brand']
        .fillna(
            df.groupby(
                ['Compartments', 'price_bin',
                 'Weight Capacity (kg)', 'Laptop Compartment',
                 'Style', 'Size', 'Material', 'Color', 'Waterproof']
            )['Brand']
            .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(
            df.groupby('Compartments')['Brand']
              .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        )
        .fillna(df['Brand'].mode().iloc[0])
)



print(df.isna().sum())


C:\Users\topi7\AppData\Local\Temp\ipykernel_27180\169136062.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['Compartments', 'price_bin'])['Weight Capacity (kg)']
C:\Users\topi7\AppData\Local\Temp\ipykernel_27180\169136062.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(
C:\Users\topi7\AppData\Local\Temp\ipykernel_27180\169136062.py:47: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.group

ValueError: invalid literal for int() with base 10: 'Medium'

In [10]:
df = df.drop(columns=['price_bin'])


In [11]:
df['Weight Capacity (kg)'].isna().sum()
df.isna().sum()



id                      0
Brand                   0
Material                0
Size                    0
Compartments            0
Laptop Compartment      0
Waterproof              0
Style                   0
Color                   0
Weight Capacity (kg)    0
Price                   0
dtype: int64

In [28]:
df = df.drop(['id'], axis=1)

for i in df.columns:
    print(df[i].unique())


KeyError: "['id'] not found in axis"

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_features = ['Compartments', 'Weight Capacity (kg)']  # Убрали Price
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [16]:
categorical_cols = ['Brand', 'Material', 'Size', 'Laptop Compartment', 'Waterproof', 'Style', 'Color']

# Разделение данных
X = df.drop(columns=['Price'])
y = df['Price']

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop='first')
X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

In [19]:
X_train_final = pd.concat([
    X_train.drop(columns=categorical_cols).reset_index(drop=True),
    pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_cols))
], axis=1)

X_test_final = pd.concat([
    X_test.drop(columns=categorical_cols).reset_index(drop=True),
    pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(categorical_cols))
], axis=1)

In [23]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='rmse'
)

model.fit(X_train_final, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'rmse'


In [24]:
y_pred = model.predict(X_test_final)


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 33.64322102532418
MSE: 1514.3863146149718
RMSE: 38.91511678788812
R² Score: 0.0014701384701960585
